# Thesis experiment pipeline
Run all cells from top to bottom. `MOCK_MODE=True` validates the complete workflow without API calls. Set it to `False` only after both API providers are verified.

In [1]:
import sys, importlib
from pathlib import Path
for _name in list(sys.modules):
    if _name == 'thesis_pipeline' or _name.startswith('thesis_pipeline.'):
        del sys.modules[_name]
importlib.invalidate_caches()
ROOT=Path.cwd().parent if Path.cwd().name=='notebooks' else Path.cwd()
sys.path.insert(0,str(ROOT))
from thesis_pipeline import *
import pandas as pd
DATASET={'name':'final_dataset_1','path':'data/final_dataset_1.csv','columns':{'instruction':'instruction','input':'input','gold':'output'}}
MOCK_MODE=False
pipeline=initialize_thesis(DATASET,RUN_MODE='full',GLOBAL_SEED=42,root=ROOT,mock=MOCK_MODE,output_dir='results/final_dataset_1')
print('Dataset:',pipeline.bundle.dataset_name)
print('Total rows:',len(pipeline.bundle.frame),'| Eligible scored rows:',pipeline.bundle.validation['eligible_cases'],'| Excluded:',pipeline.bundle.validation['excluded_cases'])
print('Excluded-case audit:',pipeline.output/'excluded_cases.csv')
print('Development cases:',len(pipeline.bundle.cases('development')),'| Test cases:',len(pipeline.bundle.cases('test')))
print('MODE:', 'OFFLINE VALIDATION (no API calls)' if MOCK_MODE else 'LIVE API')

Dataset: final_dataset_1
Total rows: 2000 | Eligible scored rows: 1995 | Excluded: 5
Excluded-case audit: C:\Users\Taghreed Al-Sharafi\Desktop\New folder\THESIS_EXPERIMENT_PIPELINE_COMPLETE_WITH_ENV\results\final_dataset_1\excluded_cases.csv
Development cases: 415 | Test cases: 1580
MODE: LIVE API


In [2]:
# One-case diagnostic
example=pipeline.bundle.cases('test').head(1)
example_pred=pipeline._mock_predictions(example,'diagnostic') if MOCK_MODE else pipeline._single_predictions(example,'diagnostic','openai',pipeline.config.gpt_model,'test')
diagnostic=example[['case_id','gold']].merge(example_pred[['case_id','prediction','status']],on='case_id')
diagnostic['correct']=diagnostic['gold'].astype(str).str.upper()==diagnostic['prediction'].astype(str).str.upper()
diagnostic

,case_id,gold,prediction,status,correct
0,f11bc208d1b0f42115dacaf22f851f57fb84366b812e4d...,C,,api_error,False


In [3]:
single_results=run_single_baselines()
print('Single-model baselines — TEST split')
single_results[['system','split','n','successful_n','failure_rate','accuracy','precision','recall','macro_f1','critical_safety_error_rate']]

Single-model baselines — TEST split


,system,split,n,successful_n,failure_rate,accuracy,precision,recall,macro_f1,critical_safety_error_rate
0,single_gpt,test,1580,0,1.0,0.0,0.0,0.0,0.0,0.0
1,single_claude,test,1580,0,1.0,0.0,0.0,0.0,0.0,0.0


In [4]:
agreement_results=run_agreement_experiments()
print('Agreement-method comparison — DEVELOPMENT split only')
agreement_results[['rank','method','split','n','successful_n','failure_rate','accuracy','precision','recall','macro_f1','critical_safety_error_rate','selected']]

Agreement-method comparison — DEVELOPMENT split only


,rank,method,split,n,successful_n,failure_rate,accuracy,precision,recall,macro_f1,critical_safety_error_rate,selected
0,1,care_consensus,development,415,0,1.0,0.0,0.0,0.0,0.0,0.0,True
1,2,jsd,development,415,0,1.0,0.0,0.0,0.0,0.0,0.0,False
2,3,kendall_w,development,415,0,1.0,0.0,0.0,0.0,0.0,0.0,False
3,4,krippendorff_alpha,development,415,0,1.0,0.0,0.0,0.0,0.0,0.0,False
4,5,ua_kalpha,development,415,0,1.0,0.0,0.0,0.0,0.0,0.0,False
5,6,vote_entropy,development,415,0,1.0,0.0,0.0,0.0,0.0,0.0,False


In [5]:
selection=select_best_agreement_method()
print('Frozen agreement method:',selection['selected_method'])
selection

Frozen agreement method: care_consensus


{'selected_method': 'care_consensus',
 'selection_split': 'development',
 'accuracy_tolerance': 0.01,
 'hierarchy': ['accuracy',
  'critical_safety_error_rate',
  'macro_f1',
  'extra_llm_calls'],
 'candidates': [{'system': 'care_consensus',
   'n': 5,
   'successful_n': 5,
   'failure_count': 0,
   'failure_rate': 0.0,
   'accuracy': 0.2,
   'accuracy_ci_low': 0.0,
   'accuracy_ci_high': 0.6,
   'critical_safety_error_rate': 0.0,
   'critical_safety_error_ci_low': 0.0,
   'critical_safety_error_ci_high': 0.0,
   'precision': 0.08333333333333333,
   'recall': 0.16666666666666666,
   'macro_f1': 0.1111111111111111,
   'f1_score': 0.1111111111111111,
   'method': 'care_consensus',
   'split': 'development',
   'extra_llm_calls': 0,
   'rank': 1,
   'selected': True},
  {'system': 'jsd',
   'n': 5,
   'successful_n': 5,
   'failure_count': 0,
   'failure_rate': 0.0,
   'accuracy': 0.2,
   'accuracy_ci_low': 0.0,
   'accuracy_ci_high': 0.6,
   'critical_safety_error_rate': 0.0,
   'critica

In [6]:
final_results=run_final_comparison()
print('Final comparison — TEST split only')
final_table=pd.DataFrame.from_dict(final_results['systems'],orient='index').drop(columns=['system'],errors='ignore').rename_axis('system').reset_index()
final_table

Final comparison — TEST split only


,system,n,successful_n,failure_count,failure_rate,accuracy,accuracy_ci_low,accuracy_ci_high,critical_safety_error_rate,critical_safety_error_ci_low,critical_safety_error_ci_high,precision,recall,macro_f1,f1_score
0,single_gpt,5,5,0,0.0,0.4,0.0,0.8,0.0,0.0,0.0,1.00,0.416667,0.583333,0.583333
1,single_claude,5,5,0,0.0,0.2,0.0,0.6,0.0,0.0,0.0,0.25,0.250000,0.250000,0.250000
2,selected_multi_agent,5,5,0,0.0,0.6,0.2,1.0,0.0,0.0,0.0,1.00,0.666667,0.750000,0.750000


In [7]:
outputs=build_thesis_outputs()
print('Generated reports:')
print(*outputs['reports'],sep='\n')
print('Generated figures:')
print(*outputs['figures'],sep='\n')

Generated reports:
C:\Users\Taghreed Al-Sharafi\Desktop\New folder\THESIS_EXPERIMENT_PIPELINE_COMPLETE_WITH_ENV\results\thesis\reports\01_single_model_results.docx
C:\Users\Taghreed Al-Sharafi\Desktop\New folder\THESIS_EXPERIMENT_PIPELINE_COMPLETE_WITH_ENV\results\thesis\reports\02_agreement_method_comparison.docx
C:\Users\Taghreed Al-Sharafi\Desktop\New folder\THESIS_EXPERIMENT_PIPELINE_COMPLETE_WITH_ENV\results\thesis\reports\03_final_single_vs_multi_comparison.docx
C:\Users\Taghreed Al-Sharafi\Desktop\New folder\THESIS_EXPERIMENT_PIPELINE_COMPLETE_WITH_ENV\results\thesis\reports\04_COMPLETE_THESIS_RESULTS.docx
Generated figures:
C:\Users\Taghreed Al-Sharafi\Desktop\New folder\THESIS_EXPERIMENT_PIPELINE_COMPLETE_WITH_ENV\results\thesis\figures\figure_1_agreement_methods.png
C:\Users\Taghreed Al-Sharafi\Desktop\New folder\THESIS_EXPERIMENT_PIPELINE_COMPLETE_WITH_ENV\results\thesis\figures\figure_2_accuracy_vs_safety.png
